# Load Pascal VOC 2012 as a 3LC Table

This notebook ingests the Pascal VOC 2012 trainval dataset as a single 3LC Table, preserving as much annotation metadata as possible.

<!-- Tags: ["object-detection", "instance-segmentation", "semantic-segmentation", "layout", "action-recognition"] -->

VOC 2012 has rich, layered annotations across four sub-tasks:

- **Main** — 20-class classification + per-object bounding boxes for all 17,125 trainval images.
- **Layout** — person-part bounding boxes (head, hand, foot, ...) for 850 images.
- **Segmentation** — pixel-perfect class + instance masks for 2,913 images, stored as paletted PNGs.
- **Action** — 11 binary action flags per person for 5,670 images.

We collapse all of this into one row per image, with columns:

- `image`, `width`, `height`, `source` (which VOC year the annotation comes from), `split` (train/val).
- `bounding_boxes` — every annotated object, with per-instance `pose`, `truncated`, `occluded`, `difficult`, `action`.
- `person_parts` — separate bbox column with head/hand/foot/... for layout images. Empty otherwise.
- `instance_segmentation` — per-instance masks decoded from `SegmentationObject` PNGs, with class labels recovered from `SegmentationClass` PNGs. Empty for non-segmented images.

Download VOC 2012 trainval from <http://host.robots.ox.ac.uk/pascal/VOC/voc2012/> and point `VOC_ROOT` below at the unpacked `VOCdevkit/VOC2012` directory.

## Install dependencies

In [ ]:
INSTALL_DEPENDENCIES = False

In [ ]:
if INSTALL_DEPENDENCIES:
    %pip install -q 3lc pillow numpy tqdm

## Imports

In [ ]:
from __future__ import annotations

import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import tlc
from PIL import Image
from tqdm import tqdm

## Project setup

In [ ]:
PROJECT_NAME = "3LC Tutorials - VOC2012"
DATASET_NAME = "VOC2012"
TABLE_NAME = "trainval"

VOC_ROOT = Path("/Users/gudbrand/Data/VOCdevkit/VOC2012")
MAX_SAMPLES: int | None = None  # set to a small int for a quick smoke-test

ANNOTATIONS_DIR = VOC_ROOT / "Annotations"
JPEG_DIR = VOC_ROOT / "JPEGImages"
SEG_CLASS_DIR = VOC_ROOT / "SegmentationClass"
SEG_OBJECT_DIR = VOC_ROOT / "SegmentationObject"
IMAGESETS_DIR = VOC_ROOT / "ImageSets"

for p in [ANNOTATIONS_DIR, JPEG_DIR, SEG_CLASS_DIR, SEG_OBJECT_DIR, IMAGESETS_DIR]:
    if not p.exists():
        raise FileNotFoundError(p)

## Fixed VOC vocabularies

VOC ships with a closed set of object classes, poses, actions and person parts. Defining them up front lets us build proper categorical schemas regardless of which subset of the data we ingest.

In [ ]:
# 20 classes, in canonical VOC order. Index 0 is reserved as background
# so the indices match the SegmentationClass palette.
VOC_CLASSES = [
    "background",
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]
CLASS_TO_IDX = {c: i for i, c in enumerate(VOC_CLASSES)}

POSES = ["Unspecified", "Frontal", "Rear", "Left", "Right"]
POSE_TO_IDX = {p: i for i, p in enumerate(POSES)}

ACTIONS = [
    "none",
    "jumping", "phoning", "playinginstrument", "reading",
    "ridingbike", "ridinghorse", "running", "takingphoto",
    "usingcomputer", "walking", "other",
]
ACTION_TO_IDX = {a: i for i, a in enumerate(ACTIONS)}

# Layout part classes appearing in <part> elements
PART_CLASSES = ["head", "hand", "foot"]
PART_TO_IDX = {p: i for i, p in enumerate(PART_CLASSES)}

SOURCES = [f"VOC{y}" for y in (2007, 2008, 2009, 2010, 2011, 2012)]
SOURCE_TO_IDX = {s: i for i, s in enumerate(SOURCES)}

SPLITS = ["train", "val", "trainval", "unknown"]
SPLIT_TO_IDX = {s: i for i, s in enumerate(SPLITS)}

## Resolve splits per image

VOC defines splits in `ImageSets/<task>/{train,val,trainval}.txt`. The task-specific files for Main, Layout, Segmentation and Action are not all in agreement — Action images, for example, are not all in the Main classification set. We aggregate across all sources to give each image a single split tag.

In [ ]:
def read_split_file(path: Path) -> set[str]:
    """Read ImageSet split files. First whitespace-separated token is the image id."""
    if not path.exists():
        return set()
    return {line.split()[0] for line in path.read_text().splitlines() if line.strip()}


def collect_split(name: str) -> set[str]:
    """Union of `<task>/<name>.txt` across the four task ImageSets."""
    ids: set[str] = set()
    for task in ("Main", "Layout", "Segmentation", "Action"):
        ids |= read_split_file(IMAGESETS_DIR / task / f"{name}.txt")
    return ids


train_ids = collect_split("train")
val_ids = collect_split("val")


def split_for(image_id: str) -> str:
    in_train = image_id in train_ids
    in_val = image_id in val_ids
    if in_train and in_val:
        return "trainval"
    if in_train:
        return "train"
    if in_val:
        return "val"
    return "unknown"


print(f"train ids: {len(train_ids)}, val ids: {len(val_ids)}")

## Parse a VOC XML annotation

Each `<object>` may carry pose, truncated, difficult, occluded, parts (`<part>`), actions (`<actions>`) and a reference point. Most fields are optional. We collect everything into a plain dict that is easy to convert into 3LC primitives below.

In [ ]:
def _int(node, tag, default=0):
    el = node.find(tag)
    return int(el.text) if el is not None and el.text is not None else default

def _float(node, tag, default=0.0):
    el = node.find(tag)
    return float(el.text) if el is not None and el.text is not None else default

def _text(node, tag, default=""):
    if node is None:
        return default
    el = node.find(tag)
    return el.text.strip() if el is not None and el.text is not None else default


def _bbox(node):
    bb = node.find("bndbox")
    return [_float(bb, "xmin"), _float(bb, "ymin"), _float(bb, "xmax"), _float(bb, "ymax")]


def _resolve_action(actions_node) -> int:
    """Pick a single action label from the 11 binary VOC action flags.

    VOC encodes actions as one tag per action with value 0/1. In practice usually only
    one is set; if multiple are set we prefer specific over `other`.
    """
    if actions_node is None:
        return ACTION_TO_IDX["none"]
    active = [a for a in ACTIONS[1:] if _int(actions_node, a) == 1]
    if not active:
        return ACTION_TO_IDX["none"]
    specific = [a for a in active if a != "other"]
    return ACTION_TO_IDX[specific[0] if specific else active[0]]


def parse_annotation(xml_path: Path) -> dict:
    root = ET.parse(xml_path).getroot()

    size = root.find("size")
    image_width = _int(size, "width")
    image_height = _int(size, "height")

    src = _text(root.find("source"), "annotation")  # e.g. "PASCAL VOC2007"
    source_year = src.replace("PASCAL ", "").strip() or "VOC2012"

    objects = []
    for obj in root.findall("object"):
        name = _text(obj, "name")
        if name not in CLASS_TO_IDX:
            # Defensive: skip unknown class names rather than corrupt the categorical mapping.
            continue

        parts = []
        for p in obj.findall("part"):
            part_name = _text(p, "name")
            if part_name in PART_TO_IDX:
                parts.append({"name": part_name, "bbox": _bbox(p)})

        objects.append({
            "name": name,
            "bbox": _bbox(obj),
            "pose": _text(obj, "pose", "Unspecified"),
            "truncated": _int(obj, "truncated"),
            "occluded": _int(obj, "occluded"),
            "difficult": _int(obj, "difficult"),
            "action": _resolve_action(obj.find("actions")),
            "parts": parts,
        })

    return {
        "filename": _text(root, "filename"),
        "width": image_width,
        "height": image_height,
        "segmented": bool(_int(root, "segmented")),
        "source": source_year,
        "objects": objects,
    }


# Sanity check
sample = parse_annotation(ANNOTATIONS_DIR / "2007_000032.xml")
print(sample["filename"], sample["width"], sample["height"], "objects:", len(sample["objects"]))

## Decode instance segmentation

VOC stores two paletted PNGs per segmented image:

- `SegmentationClass/<id>.png` — pixel value is the class index (0=background, 255=void).
- `SegmentationObject/<id>.png` — pixel value is the per-image instance id (0=background, 255=void).

Combining them gives instance-level masks with class labels. We pull the class for each instance from the most common non-void class index inside that instance's pixels — handles edge boundary mismatches cleanly.

In [ ]:
VOID_LABEL = 255


def load_instance_segmentation(image_id: str, image_width: int, image_height: int):
    """Build a SegmentationMasks for the given image, or None if no masks exist."""
    cls_path = SEG_CLASS_DIR / f"{image_id}.png"
    obj_path = SEG_OBJECT_DIR / f"{image_id}.png"
    if not (cls_path.exists() and obj_path.exists()):
        return None

    class_arr = np.array(Image.open(cls_path))
    obj_arr = np.array(Image.open(obj_path))

    instance_ids = [int(i) for i in np.unique(obj_arr) if i != 0 and i != VOID_LABEL]
    if not instance_ids:
        return None

    masks, labels = [], []
    for inst_id in instance_ids:
        mask = obj_arr == inst_id
        # Resolve class from class mask: most common non-void label inside the instance.
        cls_pixels = class_arr[mask]
        cls_pixels = cls_pixels[cls_pixels != VOID_LABEL]
        if cls_pixels.size == 0:
            continue
        vals, counts = np.unique(cls_pixels, return_counts=True)
        cls_idx = int(vals[counts.argmax()])
        masks.append(mask.astype(np.uint8))
        labels.append(cls_idx)

    if not masks:
        return None

    return tlc.data_types.SegmentationMasks(
        image_width=image_width,
        image_height=image_height,
        masks=np.stack(masks, axis=-1),
        labels=np.array(labels, dtype=np.int32),
    )

## Build a row

Putting it all together: each row covers one image and exposes annotations from every available task.

In [ ]:
def build_bounding_boxes(ann):
    bbs = tlc.data_types.BoundingBoxes2D.create_empty(
        image_width=ann["width"], image_height=ann["height"]
    )
    for obj in ann["objects"]:
        bbs.add_instance(
            bounding_box=obj["bbox"],
            label=CLASS_TO_IDX[obj["name"]],
            per_instance_extras={
                "pose": POSE_TO_IDX.get(obj["pose"], POSE_TO_IDX["Unspecified"]),
                "truncated": obj["truncated"],
                "occluded": obj["occluded"],
                "difficult": obj["difficult"],
                "action": obj["action"],
            },
        )
    return bbs


def build_person_parts(ann):
    bbs = tlc.data_types.BoundingBoxes2D.create_empty(
        image_width=ann["width"], image_height=ann["height"]
    )
    for obj in ann["objects"]:
        for part in obj["parts"]:
            bbs.add_instance(bounding_box=part["bbox"], label=PART_TO_IDX[part["name"]])
    return bbs


def build_empty_segmentation(image_width: int, image_height: int):
    return tlc.data_types.SegmentationMasks(
        image_width=image_width,
        image_height=image_height,
        masks=np.zeros((image_height, image_width, 0), dtype=np.uint8),
        labels=np.zeros((0,), dtype=np.int32),
    )


def build_row(image_id: str):
    xml_path = ANNOTATIONS_DIR / f"{image_id}.xml"
    ann = parse_annotation(xml_path)

    image_url = tlc.Url(JPEG_DIR / ann["filename"]).to_relative().to_str()
    seg = load_instance_segmentation(image_id, ann["width"], ann["height"]) if ann["segmented"] else None

    has_layout = any(obj["parts"] for obj in ann["objects"])

    return {
        "image": image_url,
        "image_id": image_id,
        "width": ann["width"],
        "height": ann["height"],
        "source": SOURCE_TO_IDX.get(ann["source"], SOURCE_TO_IDX["VOC2012"]),
        "split": SPLIT_TO_IDX[split_for(image_id)],
        "num_objects": len(ann["objects"]),
        "has_segmentation": int(seg is not None),
        "has_layout": int(has_layout),
        "bounding_boxes": build_bounding_boxes(ann),
        "person_parts": build_person_parts(ann),
        "instance_segmentation": seg if seg is not None else build_empty_segmentation(ann["width"], ann["height"]),
    }


# Preview one row to verify everything is wired up
preview = build_row("2007_000032")
print({k: v for k, v in preview.items() if not hasattr(v, "num_instances")})
print("bboxes:", preview["bounding_boxes"].num_instances)
print("parts:", preview["person_parts"].num_instances)
print("seg masks:", preview["instance_segmentation"].num_instances)

## Define schemas

We use `CategoricalLabelSchema` for image-level categorical columns, and the per-instance schemas inside `BoundingBoxes2D.schema(...)` so that pose/action filter UIs render the human-readable strings. Annotation columns get their schema-builders directly from the matching dataclass — `BoundingBoxes2D.schema()` and `SegmentationMasks.schema()` — keeping the runtime type and column description paired.

In [ ]:
def _map(values):
    return {i: tlc.MapElement(v) for i, v in enumerate(values)}


schemas = {
    "image": tlc.schemas.ImageSchema(sample_type="url"),
    # "image_id": tlc.schemas.StringSchema(),
    # "width": tlc.schemas.Int32Schema(),
    # "height": tlc.schemas.Int32Schema(),
    "source": tlc.schemas.CategoricalLabelSchema(classes=SOURCES),
    "split": tlc.schemas.CategoricalLabelSchema(classes=SPLITS),
    "num_objects": tlc.schemas.Int32Schema(),
    "has_segmentation": tlc.schemas.Int32Schema(),
    "has_layout": tlc.schemas.Int32Schema(),
    "bounding_boxes": tlc.data_types.BoundingBoxes2D.schema(
        classes=VOC_CLASSES,
        per_instance_schemas={
            "pose": tlc.schemas.CategoricalLabelSchema(classes=POSES),
            "truncated": tlc.schemas.Int32Schema(),
            "occluded": tlc.schemas.Int32Schema(),
            "difficult": tlc.schemas.Int32Schema(),
            "action": tlc.schemas.CategoricalLabelSchema(classes=ACTIONS),
        },
    ),
    "person_parts": tlc.data_types.BoundingBoxes2D.schema(classes=PART_CLASSES),
    "instance_segmentation": tlc.data_types.SegmentationMasks.schema(classes=VOC_CLASSES),
}

print(f"Built {len(schemas)} schemas")

## Write the table

Iterate over every XML annotation, build a row, and stream it through `TableWriter`.

In [ ]:
image_ids = sorted(p.stem for p in ANNOTATIONS_DIR.glob("*.xml"))
if MAX_SAMPLES is not None:
    image_ids = image_ids[:MAX_SAMPLES]
print(f"Ingesting {len(image_ids)} images")

table_writer = tlc.TableWriter(
    table_name=TABLE_NAME,
    dataset_name=DATASET_NAME,
    project_name=PROJECT_NAME,
    schema=schemas,
)

for image_id in tqdm(image_ids, desc="Writing rows"):
    table_writer.add_row(build_row(image_id))

table = table_writer.finalize()
print(f"Created table with {len(table)} rows")
print(f"Table URL: {table.url}")

In [ ]:
table[1]

In [ ]:
table.url

In [ ]:
# train_split = tlc.FilteredTable(
#     input_table_url=table,
#     filter_criterion=tlc.NumericRangeFilterCriterion("split", -1, 1),
#     url=table.url.parent / "train",
# )

# print(len(train_split))

In [ ]:
# val_split = tlc.FilteredTable(
#     input_table_url=table,
#     filter_criterion=tlc.NumericRangeFilterCriterion("split", 0, 2),
#     url=table.url.parent / "val",
# )

# print(len(val_split))